# Homework Option A: Build a Toy 2D Diffusion Model

Time: up to 60 minutes. Runs on CPU. No GPU, Drive, or API key needed.

This notebook builds the same diffusion model concept from Part 1 and Part 2
of the workshop, but for 2D points instead of images.
Look for `FILL_IN` and complete the two formulas.
See `Homework_OptionA_ToyDiffusion.md` for full instructions.

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
print('Setup ready.')

---

## Step 1: The Dataset

`make_moons` generates 2D points shaped like two interleaving crescents.
We normalize the points so they have roughly zero mean and unit scale,
similar to scaling images to [-1, 1] in the workshop.

In [ ]:
X, _ = make_moons(n_samples=2000, noise=0.05)
X = torch.tensor(X, dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0) * 0.5

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], s=5, alpha=0.5)
plt.title('Training data: two moons')
plt.axis('equal')
plt.show()

---

## Step 2: Noise Schedule

Same idea as the workshop, just a smaller T since this is a simpler problem.

In [ ]:
T = 100
B = torch.linspace(1e-4, 0.02, T)
a = 1 - B
a_bar = torch.cumprod(a, dim=0)
sqrt_a_bar = torch.sqrt(a_bar)
sqrt_one_minus_a_bar = torch.sqrt(1 - a_bar)
sqrt_a_inv = torch.sqrt(1 / a)
pred_noise_coeff = (1 - a) / sqrt_one_minus_a_bar

print('Noise schedule ready.')

### FILL_IN 1: Closed-Form Forward Diffusion

Same formula as Part 1, Exercise 2 in the workshop, just applied to 2D points
instead of images. `x_0` has shape `[batch, 2]`.

In [ ]:
def q(x_0, t):
    noise = torch.randn_like(x_0)
    sqrt_a_bar_t    = sqrt_a_bar[t][:, None]
    sqrt_1m_a_bar_t = sqrt_one_minus_a_bar[t][:, None]
    # FILL_IN: combine x_0 and noise using the two coefficients above
    x_t = FILL_IN
    return x_t, noise

<details>
<summary><b>Click to show the solution</b></summary>

```python
x_t = sqrt_a_bar_t * x_0 + sqrt_1m_a_bar_t * noise
```
</details>

---

## Step 3: The Model

A tiny MLP. It takes a 2D point plus a normalized timestep,
and predicts the noise that was added.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        t_norm = (t.float() / T).unsqueeze(1)
        inp = torch.cat([x, t_norm], dim=1)
        return self.net(inp)

model = TinyMLP()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-3)
STEPS = 2000

for step in range(STEPS):
    optimizer.zero_grad()
    idx = torch.randint(0, len(X), (256,))
    x0  = X[idx]
    t   = torch.randint(0, T, (256,))
    x_noisy, noise = q(x0, t)
    pred = model(x_noisy, t)
    loss = F.mse_loss(pred, noise)
    loss.backward()
    optimizer.step()
    if step % 400 == 0:
        print(f'Step {step}/{STEPS} | loss: {loss.item():.4f}')

print('Training done.')

### FILL_IN 2: Reverse Diffusion

Same formula as Part 2, Exercise 1 in the workshop.

In [ ]:
@torch.no_grad()
def reverse_q(x_t, t, e_t):
    coeff        = pred_noise_coeff[t][:, None]
    sqrt_a_inv_t = sqrt_a_inv[t][:, None]
    # FILL_IN: combine x_t, e_t, coeff, and sqrt_a_inv_t
    u_t = FILL_IN
    if t[0] == 0:
        return u_t
    B_prev = B[t - 1][:, None]
    return u_t + torch.sqrt(B_prev) * torch.randn_like(x_t)

<details>
<summary><b>Click to show the solution</b></summary>

```python
u_t = sqrt_a_inv_t * (x_t - coeff * e_t)
```
</details>

---

## Step 4: Generate and Compare

In [ ]:
@torch.no_grad()
def sample(n=500):
    x = torch.randn(n, 2)
    for i in range(T - 1, -1, -1):
        t = torch.full((n,), i)
        e_t = model(x, t)
        x = reverse_q(x, t, e_t)
    return x

generated = sample()

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=5, alpha=0.3, label='real')
plt.scatter(generated[:, 0], generated[:, 1], s=5, alpha=0.3, label='generated')
plt.legend()
plt.axis('equal')
plt.title('Real vs. generated points')
plt.show()

---

## Step 5: Try a Different Shape

Replace the dataset with a different shape. Two options:

```python
# Option 1: circles instead of moons
from sklearn.datasets import make_circles
X, _ = make_circles(n_samples=2000, noise=0.05, factor=0.5)
X = torch.tensor(X, dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0) * 0.5
```

```python
# Option 2: a heart shape
t_vals = torch.linspace(0, 2 * np.pi, 2000)
x_heart = 16 * torch.sin(t_vals) ** 3
y_heart = 13*torch.cos(t_vals) - 5*torch.cos(2*t_vals) - 2*torch.cos(3*t_vals) - torch.cos(4*t_vals)
X = torch.stack([x_heart, y_heart], dim=1)
X = X + torch.randn_like(X) * 0.3   # add a little noise for variety
X = (X - X.mean(0)) / X.std(0) * 0.5
```

Copy one of these into a new cell below, re-run the training cell, and sample again.

In [ ]:
# TODO: paste your chosen dataset code here, then re-run training above


---

## Report

**1. Did your model learn the two-moons shape well? What did the generated points look like?**

_(your answer here)_

**2. What shape did you try in Step 5? Did the model learn it as well as the moons?**

_(your answer here)_

**3. What happens if you reduce STEPS a lot (for example to 200)? Try it and describe what you see.**

_(your answer here)_